# Importando Bibliotecas

In [1]:
# Seleção de diretórios
import os
try:
    # Configurando diretório de trabalho para 'aulas_teoricas'
    os.chdir('aulas_teoricas')
except FileNotFoundError:
    pass

# Carregamento de dados
from joblib import load, dump

# Processamento de dados
import numpy as np
import pandas as pd
from util import utils as ut

# Graficos
import seaborn as sns
import matplotlib.pyplot as plt

# Importando Dados

In [2]:
# Verificando os arquivos no diretório 'data/df_estruturada'
os.listdir("./data/df_estruturada")

['08_aula_09_X_test',
 '08_aula_09_X_train',
 '08_aula_09_X_valid',
 '08_aula_09_y_test',
 '08_aula_09_y_train',
 '08_aula_09_y_valid']

In [3]:
# Importando os dados de treino
df_train = pd.concat([
    load("./data/df_estruturada/08_aula_09_X_train"),
    load("./data/df_estruturada/08_aula_09_y_train")
], axis=1)
df_train.head()

,TP53,IFN-γ,CD3_2,CD8,PPD_log,correlatas,FOXP3,classe
541,0.060824,0.156681,2054.750574,416.256426,1.728509,0.274165,Alterado,imunodeficiencia
440,0.049927,0.277029,1853.437984,561.318387,1.240507,0.222972,Alterado,imunodeficiencia
482,0.020794,0.169621,3712.239299,1314.103239,1.470395,0.468595,Normal,leucemia
422,0.098734,0.117708,1864.705992,802.282545,1.574182,0.288605,Alterado,imunodeficiencia
778,0.193195,0.109547,2926.017038,1524.468444,2.400111,0.466135,Normal,leucemia


In [4]:
df_train['classe'].value_counts(normalize=True)

cancer              0.285714
BCG                 0.258571
imunodeficiencia    0.238571
leucemia            0.217143
Name: classe, dtype: float64

# Validação cruzada e busca de hiperparâmetros

## K-NN

### Treino

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [12]:
# 2. Separação em X (features) e y (target)
X = df_train.drop(columns=['classe'])
y = df_train['classe']

# Identificação das colunas por tipo
num_features = ['TP53','IFN-γ','CD3_2','CD8','PPD_log','correlatas']
cat_features = ['FOXP3']

# 3. Divisão em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Construção dos Preprocessadores específicos por tipo de coluna
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')
# Nota: 'drop="first"' evita a armadilha das variáveis dummy (multicolinearidade).
# 'handle_unknown="ignore"' impede erros se uma categoria nova aparecer no teste/validação.

# 5. Agrupamento das transformações com ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

# 6. Pipeline Principal
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('knn', KNeighborsClassifier())
])

# 7. Definição da Grade de Hiperparâmetros (Grid Search)
param_grid = {
    'knn__n_neighbors': list(range(1, 7, 2)), # Ajuste o intervalo conforme o tamanho do seu dataset
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

# 8. Execução do Grid Search com Validação Cruzada
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,  # Ajustado para 3 devido ao tamanho pequeno do dataset do exemplo (use 5 ou 10 em dados reais)
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("--- Treinando modelo com Pipeline de pré-processamento ---")
grid_search.fit(X_train, y_train)

# 9. Exibição dos Resultados
print("\n--- Melhores Resultados do Grid Search ---")
print(f"Melhores hiperparâmetros: {grid_search.best_params_}")
print(f"Melhor acurácia (CV): {grid_search.best_score_:.4f}")

# 10. Avaliação no conjunto de Teste
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- Desempenho no Conjunto de Teste ---")
print(f"Acurácia final: {accuracy_score(y_test, y_pred):.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred))

--- Treinando modelo com Pipeline de pré-processamento ---
Fitting 3 folds for each of 12 candidates, totalling 36 fits

--- Melhores Resultados do Grid Search ---
Melhores hiperparâmetros: {'knn__metric': 'manhattan', 'knn__n_neighbors': 5, 'knn__weights': 'uniform'}
Melhor acurácia (CV): 0.7750

--- Desempenho no Conjunto de Teste ---
Acurácia final: 0.7357

Relatório de Classificação:
                  precision    recall  f1-score   support

             BCG       1.00      1.00      1.00        36
          cancer       0.53      0.62      0.57        40
imunodeficiencia       1.00      1.00      1.00        34
        leucemia       0.35      0.27      0.30        30

        accuracy                           0.74       140
       macro avg       0.72      0.72      0.72       140
    weighted avg       0.73      0.74      0.73       140



# Avalia Modelos

In [5]:
from sklearn import svm

from src.avalia_modelo import Avalia_modelo

In [ ]:
# SVC for classification
clf = svm.SVC(kernel='linear') # 'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'
clf.fit(X_train, y_train)

SVC(kernel='linear')

In [ ]:
avalia_modelo = Avalia_modelo(
    df=df_5_test,
    target=target,
    indep=indep,
    modelo=clf
)